In [ ]:
import pandas as pd
package_data = pd.read_csv("../package_data.csv")

In [2]:
import requests
import csv
import time
from urllib.parse import quote
from requests.exceptions import RequestException, Timeout, ConnectionError

def make_request(url, timeout=30, max_retries=3):
    """Make a request with timeout and retries."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=timeout)
            if resp.status_code == 429:  # rate limited
                time.sleep(5)
                continue
            return resp, None
        except Timeout:
            error = f"Timeout after {timeout}s (attempt {attempt + 1}/{max_retries})"
            if attempt < max_retries - 1:
                time.sleep(2)
        except ConnectionError as e:
            error = f"Connection error: {str(e)[:100]} (attempt {attempt + 1}/{max_retries})"
            if attempt < max_retries - 1:
                time.sleep(5)
        except RequestException as e:
            error = f"Request error: {str(e)[:100]}"
            break
    return None, error

def get_version_dependents(ecosystem, package, version, delay=0.1):
    """Get direct dependent count for a specific package version."""
    encoded_package = quote(package, safe='')
    encoded_version = quote(version, safe='')
    url = f"https://api.deps.dev/v3alpha/systems/{ecosystem}/packages/{encoded_package}/versions/{encoded_version}:dependents"
    
    time.sleep(delay)
    resp, error = make_request(url)
    
    if error:
        return None, error
    if resp.ok:
        data = resp.json()
        return data.get('directDependentCount', 0), None
    else:
        return None, f"HTTP {resp.status_code}: {resp.text[:200]}"

def get_all_versions_dependents(ecosystem, package):
    """Get dependent counts for all versions of a package."""
    encoded_package = quote(package, safe='')
    url = f"https://api.deps.dev/v3alpha/systems/{ecosystem}/packages/{encoded_package}"
    
    resp, error = make_request(url)
    
    if error:
        return [], [{'ecosystem': ecosystem, 'package': package, 'version': '', 'error': error}]
    if not resp.ok:
        error_msg = f"HTTP {resp.status_code}: {resp.text[:200]}"
        return [], [{'ecosystem': ecosystem, 'package': package, 'version': '', 'error': error_msg}]
    
    data = resp.json()
    versions = [v['versionKey']['version'] for v in data.get('versions', [])]
    
    results = []
    failures = []
    
    for version in versions:
        count, error = get_version_dependents(ecosystem, package, version)
        if error is None:
            results.append({
                'ecosystem': ecosystem,
                'package': package,
                'version': version,
                'dependents': count
            })
        else:
            failures.append({
                'ecosystem': ecosystem,
                'package': package,
                'version': version,
                'error': error
            })
    
    return results, failures

def save_to_csv(data, filename, fieldnames, write_header=False):
    """Append data to CSV file."""
    if not data:
        return
    mode = 'w' if write_header else 'a'
    with open(filename, mode, newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerows(data)

def process_packages(packages, output_file='dependents.csv', failures_file='failures.csv', save_every=50):
    """Process packages and save to CSV periodically."""
    all_results = []
    all_failures = []
    first_save = True
    
    for i, (ecosystem, package) in enumerate(packages):
        print(f"[{i+1}/{len(packages)}] Processing {package}...")
        results, failures = get_all_versions_dependents(ecosystem, package)
        all_results.extend(results)
        all_failures.extend(failures)
        
        if failures:
            print(f"  WARNING: {len(failures)} failures for {package}")
        
        # Save every N packages
        if (i + 1) % save_every == 0:
            save_to_csv(all_results, output_file, 
                       fieldnames=['ecosystem', 'package', 'version', 'dependents'],
                       write_header=first_save)
            save_to_csv(all_failures, failures_file,
                       fieldnames=['ecosystem', 'package', 'version', 'error'],
                       write_header=first_save)
            print(f"  Saved {len(all_results)} rows, {len(all_failures)} failures")
            all_results = []
            all_failures = []
            first_save = False
    
    # Save any remaining results
    if all_results or all_failures:
        save_to_csv(all_results, output_file,
                   fieldnames=['ecosystem', 'package', 'version', 'dependents'],
                   write_header=first_save)
        save_to_csv(all_failures, failures_file,
                   fieldnames=['ecosystem', 'package', 'version', 'error'],
                   write_header=first_save)
        print(f"  Saved final {len(all_results)} rows, {len(all_failures)} failures")

# Usage
# unique_packages = package_data[['System', 'package_name']].drop_duplicates()
# packages = list(zip(unique_packages['System'], unique_packages['package_name']))

# # Load already processed packages
# already_done = pd.read_csv('dependents.csv')[['ecosystem', 'package']].drop_duplicates()
# already_done_set = set(zip(already_done['ecosystem'], already_done['package']))

# remaining_packages = [(eco, pkg) for eco, pkg in packages if (eco, pkg) not in already_done_set]

# print(f"Total: {len(packages)}, Already done: {len(already_done_set)}, Remaining: {len(remaining_packages)}")

# process_packages(remaining_packages, output_file='dependents_run2.csv', failures_file='failures_run2.csv', save_every=10)

In [ ]:
## if failure reason is timeout, rerun
import pandas as pd

# 1. Load the failures from the previous run
failures_df = pd.read_csv('failures_run2.csv')

# 2. Filter for rows where the error contains 'Timeout'
# We use case=False just to be safe
timeout_failures = failures_df[failures_df['error'].str.contains('timeout', case=False, na=False)]

# 3. Get unique ecosystem/package pairs to retry
# We retry the whole package to get all versions correctly
unique_retries = timeout_failures[['ecosystem', 'package']].drop_duplicates()
packages_to_retry = list(zip(unique_retries['ecosystem'], unique_retries['package']))

print(f"Found {len(timeout_failures)} timeout rows across {len(packages_to_retry)} unique packages.")

if packages_to_retry:
    # 4. Run the process again
    # We use new filenames to avoid mixing data until we're sure it worked
    process_packages(
        packages_to_retry, 
        output_file='dependents_retried.csv', 
        failures_file='failures_retried.csv', 
        save_every=5  # Saving more frequently for smaller batches
    )
else:
    print("No timeout errors found to retry.")

## Only 11 rows have timeout failures that we need to rerun the script for. 

In [3]:
## if failure reason is timeout, rerun
import pandas as pd

# 1. Load the failures from the previous run
failures_df = pd.read_csv('failures_retried.csv')

# 2. Filter for rows where the error contains 'Timeout'
# We use case=False just to be safe
timeout_failures = failures_df[failures_df['error'].str.contains('timeout', case=False, na=False)]

# 3. Get unique ecosystem/package pairs to retry
# We retry the whole package to get all versions correctly
unique_retries = timeout_failures[['ecosystem', 'package']].drop_duplicates()
packages_to_retry = list(zip(unique_retries['ecosystem'], unique_retries['package']))

print(f"Found {len(timeout_failures)} timeout rows across {len(packages_to_retry)} unique packages.")

if packages_to_retry:
    # 4. Run the process again
    # We use new filenames to avoid mixing data until we're sure it worked
    process_packages(
        packages_to_retry, 
        output_file='dependents_retried_2.csv', 
        failures_file='failures_retried_2.csv', 
        save_every=5  # Saving more frequently for smaller batches
    )
else:
    print("No timeout errors found to retry.")

Found 11 timeout rows across 3 unique packages.
[1/3] Processing helix-fhir-client-sdk...
[2/3] Processing @pulumi/pulumi...
[3/3] Processing browsertime...
  Saved final 5896 rows, 19 failures


**Combined the successful outputs from running the script in dependent_count.csv.**

## I am missing dependent information on repositories added 1/1/26. Need to determine which repositories need depenendent information and collect data for thos repositories.

In [7]:
import pandas as pd
import os
package_data = pd.read_csv("../../package_data.csv")

In [5]:
print(os.getcwd())

/Users/jillmarley/Scorecard-Metrics-Over-Time/control_variables/direct_dependents


In [ ]:
import requests
import csv
import time
from urllib.parse import quote
from requests.exceptions import RequestException, Timeout, ConnectionError

def make_request(url, timeout=30, max_retries=3):
    """Make a request with timeout and retries."""
    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=timeout)
            if resp.status_code == 429:  # rate limited
                time.sleep(5)
                continue
            return resp, None
        except Timeout:
            error = f"Timeout after {timeout}s (attempt {attempt + 1}/{max_retries})"
            if attempt < max_retries - 1:
                time.sleep(2)
        except ConnectionError as e:
            error = f"Connection error: {str(e)[:100]} (attempt {attempt + 1}/{max_retries})"
            if attempt < max_retries - 1:
                time.sleep(5)
        except RequestException as e:
            error = f"Request error: {str(e)[:100]}"
            break
    return None, error

def get_version_dependents(ecosystem, package, version, delay=0.1):
    """Get direct dependent count for a specific package version."""
    encoded_package = quote(package, safe='')
    encoded_version = quote(version, safe='')
    url = f"https://api.deps.dev/v3alpha/systems/{ecosystem}/packages/{encoded_package}/versions/{encoded_version}:dependents"
    
    time.sleep(delay)
    resp, error = make_request(url)
    
    if error:
        return None, error
    if resp.ok:
        data = resp.json()
        return data.get('directDependentCount', 0), None
    else:
        return None, f"HTTP {resp.status_code}: {resp.text[:200]}"

def get_all_versions_dependents(ecosystem, package):
    """Get dependent counts for all versions of a package."""
    encoded_package = quote(package, safe='')
    url = f"https://api.deps.dev/v3alpha/systems/{ecosystem}/packages/{encoded_package}"
    
    resp, error = make_request(url)
    
    if error:
        return [], [{'ecosystem': ecosystem, 'package': package, 'version': '', 'error': error}]
    if not resp.ok:
        error_msg = f"HTTP {resp.status_code}: {resp.text[:200]}"
        return [], [{'ecosystem': ecosystem, 'package': package, 'version': '', 'error': error_msg}]
    
    data = resp.json()
    versions = [v['versionKey']['version'] for v in data.get('versions', [])]
    
    results = []
    failures = []
    
    for version in versions:
        count, error = get_version_dependents(ecosystem, package, version)
        if error is None:
            results.append({
                'ecosystem': ecosystem,
                'package': package,
                'version': version,
                'dependents': count
            })
        else:
            failures.append({
                'ecosystem': ecosystem,
                'package': package,
                'version': version,
                'error': error
            })
    
    return results, failures

def save_to_csv(data, filename, fieldnames, write_header=False):
    """Append data to CSV file."""
    if not data:
        return
    mode = 'w' if write_header else 'a'
    with open(filename, mode, newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerows(data)

def process_packages(packages, output_file='dependents.csv', failures_file='failures.csv', save_every=50):
    """Process packages and save to CSV periodically."""
    all_results = []
    all_failures = []
    first_save = True
    
    for i, (ecosystem, package) in enumerate(packages):
        print(f"[{i+1}/{len(packages)}] Processing {package}...")
        results, failures = get_all_versions_dependents(ecosystem, package)
        all_results.extend(results)
        all_failures.extend(failures)
        
        if failures:
            print(f"  WARNING: {len(failures)} failures for {package}")
        
        # Save every N packages
        if (i + 1) % save_every == 0:
            save_to_csv(all_results, output_file, 
                       fieldnames=['ecosystem', 'package', 'version', 'dependents'],
                       write_header=first_save)
            save_to_csv(all_failures, failures_file,
                       fieldnames=['ecosystem', 'package', 'version', 'error'],
                       write_header=first_save)
            print(f"  Saved {len(all_results)} rows, {len(all_failures)} failures")
            all_results = []
            all_failures = []
            first_save = False
    
    # Save any remaining results
    if all_results or all_failures:
        save_to_csv(all_results, output_file,
                   fieldnames=['ecosystem', 'package', 'version', 'dependents'],
                   write_header=first_save)
        save_to_csv(all_failures, failures_file,
                   fieldnames=['ecosystem', 'package', 'version', 'error'],
                   write_header=first_save)
        print(f"  Saved final {len(all_results)} rows, {len(all_failures)} failures")

# Usage
unique_packages = package_data[['System', 'package_name']].drop_duplicates()
packages = list(zip(unique_packages['System'], unique_packages['package_name']))

# # Load already processed packages
already_done = pd.read_csv('dependent_count.csv')[['ecosystem', 'package']].drop_duplicates()
already_done_set = set(zip(already_done['ecosystem'], already_done['package']))

remaining_packages = [(eco, pkg) for eco, pkg in packages if (eco, pkg) not in already_done_set]

print(f"Total: {len(packages)}, Already done: {len(already_done_set)}, Remaining: {len(remaining_packages)}")

process_packages(remaining_packages, output_file='dependents_run3.csv', failures_file='failures_run3.csv', save_every=10)

## How many repositories have dependent count? **16,442** repositories

In [1]:
import pandas as pd
dependents = pd.read_csv('dependent_count.csv')